
# QCE26 Tutorial: D1Q2 Quantum Lattice Boltzmann Method for the 1D Advection-Diffusion Equation

## Main target

In this notebook we show how the advection-diffusion problem can be solved using the Quantum Lattice Boltzmann Method. We focus on data logging and provenance that guide quantum algorithm development and execution.

### Contents

1. [The advection-diffusion problem](#sec-problem) — the PDE and its analytical solution
2. [Classical D1Q2 LBM](#sec-lbm) — collision, streaming, and validation against the analytical solution
3. [Quantum LBM circuit](#sec-qlbm) — amplitude encoding, streaming on qubits, and statevector verification
4. [Experiment tracking with MLflow](#sec-tracking) — logging parameters, metrics, and artifacts
5. [Sweep & analysis](#sec-sweep) — shot count and noise sweeps, error analysis, Gaussian fitting


## Setup

Install dependencies:
```
pip install numpy matplotlib pylatexenc mlflow scipy qiskit qiskit-aer ipykernel
```

After running the notebook, launch the MLflow UI to explore logged runs:
```
MLFLOW_ALLOW_FILE_STORE=true mlflow ui --backend-store-uri mlruns --port 5555
```
Then open [http://localhost:5555](http://127.0.0.1:5555) in your browser.

<a id="sec-problem"></a>

---


## 1. The Advection-Diffusion Problem

Our goal is to solve the one dimensional advection-diffusion problem described by the equation
$$
\begin{align}
\frac{\partial   \rho(x,t)}{\partial t}+u\frac{\partial   \rho(x,t)}{\partial x}
= D\frac{{\partial}^2   \rho(x,t)}{\partial x^2}
\end{align}
$$

where
- $\rho(x,t)$ - Density or Concentration,
- $D$ - Diffusion Coefficient,
- $u$ - Advection velocity,
- $x$ - Position,
- $t$ - Time


In advection–diffusion process, both advection and diffusion take place simultaneously. In general, the advection-diffusion equation describes how a quantity is simultaneously carried by a moving fluid and spread by diffusion. 

We consider a case where the function $\rho$ behaves periodically in our lattice.

### How LBM solves advection-diffusion

<details>
<summary>D1Q2 Lattice Boltzmann — collision, streaming, and the link to the PDE</summary>

Instead of discretising the PDE directly, LBM tracks two populations per lattice site — right-movers $f_0$ and left-movers $f_1$ — and repeats two steps:

1. **Collision** — relax populations toward a local equilibrium:
$$
\hat{f}_v = f_v + \frac{\Delta t}{\tau}\bigl(f_v^{eq} - f_v\bigr), \qquad
f_v^{eq} = \tfrac{1}{2}\rho\,(1 + c_v\, u)
$$

2. **Streaming** — shift each population one site in its own direction:
$$
f_0(x{+}1,\, t{+}1) = \hat{f}_0(x,t), \qquad
f_1(x{-}1,\, t{+}1) = \hat{f}_1(x,t)
$$

The macroscopic density is $\rho = f_0 + f_1$.
The relaxation time $\tau$ controls diffusion: $D = \tau - \tfrac{1}{2}$ (in lattice units where $\Delta x = \Delta t = 1$).

A Chapman–Enskog expansion shows that these local rules recover the advection-diffusion equation at the macroscopic scale.

**First-step simplification:** when populations start at equilibrium and $\tau = 1$, collision is a no-op and only streaming matters — this is the case we implement on the quantum circuit.
</details>


In [ ]:
import os

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"  # permits the local ./mlruns fallback

import matplotlib.pyplot as plt
import mlflow
import numpy as np
from IPython.display import HTML
from matplotlib.animation import FuncAnimation, PillowWriter
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error


In [ ]:
# d(rho)/dt = -u d(rho)/dx + D d2(rho)/dx2


def advection(rho, dx, u):
    return -u * (np.roll(rho, -1) - np.roll(rho, 1)) / (2 * dx)


def diffusion(rho, dx, D):
    return D * (np.roll(rho, -1) - 2 * rho + np.roll(rho, 1)) / dx**2


def rhs(rho, dx, u, D):
    return advection(rho, dx, u) + diffusion(rho, dx, D)


# grid (periodic)
L, nx = 12.0, 240
x = np.linspace(0, L, nx, endpoint=False)
dx = x[1] - x[0]

u, D = 1.0, 0.05
X0, SIGMA0 = 2.0, 0.4

dt, nsteps = 0.01, 500
t = np.arange(nsteps + 1) * dt


def analytic(x, t, n_images=3):
    sigma = np.sqrt(SIGMA0**2 + 2 * D * t)
    centre = (X0 + u * t) % L
    total = sum(np.exp(-((x - centre + k * L) ** 2) / (2 * sigma**2))
                for k in range(-n_images, n_images + 1))
    return (SIGMA0 / sigma) * total


history = np.array([analytic(x, tk) for tk in t])
rho0 = history[0]


In [ ]:
GIF = "advection_diffusion.gif"

fig, ax = plt.subplots(figsize=(10, 4))
(line,) = ax.plot(x, history[0], lw=2.5)
ax.set_xlim(0, L)
ax.set_ylim(0, history.max() * 1.05)
ax.set_xlabel("x")
ax.set_ylabel("rho")
title = ax.set_title("t = 0.00")
plt.tight_layout()


def frame(i):
    line.set_ydata(history[i])
    title.set_text(f"t = {t[i]:.2f}")
    return line, title


GIF_FPS = 25
GIF_SECONDS = 8

stride = max(1, round(len(history) / (GIF_FPS * GIF_SECONDS)))
gif_frames = range(0, len(history), stride)

anim = FuncAnimation(fig, frame, frames=gif_frames,
                     interval=1000 / GIF_FPS, blit=True)
anim.save(GIF, writer=PillowWriter(fps=GIF_FPS))
plt.close(fig)

print(f"{GIF}: {len(gif_frames)} frames at {GIF_FPS} fps")
HTML(f'<img src="{GIF}">')


<a id="sec-lbm"></a>

---

## 2. Classical D1Q2 LBM


In [ ]:
# D1Q2: two populations per site
#   f0 -> right-moving, lattice velocity c0 = +1
#   f1 -> left-moving,  lattice velocity c1 = -1
# weights w0 = w1 = 1/2, speed of sound cs = 1, dt = 1


def equilibrium(rho, u):
    # f_v^eq = w_v rho (1 + c_v u / cs^2)
    return 0.5 * rho * (1 + u), 0.5 * rho * (1 - u)


def collide(f0, f1, u, tau):
    # BGK relaxation: f_v + (dt/tau)(f_v^eq - f_v)
    f0_eq, f1_eq = equilibrium(f0 + f1, u)
    return f0 + (f0_eq - f0) / tau, f1 + (f1_eq - f1) / tau


def stream(f0, f1):
    # periodic shift by one site, in each population's own direction
    return np.roll(f0, +1), np.roll(f1, -1)


def density(f0, f1):
    # rho = sum_v f_v
    return f0 + f1


In [ ]:
TAU = 1  # relaxation time, must be > 0.5;  D_lattice = TAU - 0.5


def lbm_solve(N, tau=TAU):
    # physical units -> lattice units (dx = dt = 1 on the lattice)
    dx_l = L / N
    dt_l = (tau - 0.5) * dx_l**2 / D  # diffusive scaling: dt ~ dx^2 keeps tau fixed as N grows
    u_l = u * dt_l / dx_l             # advection velocity in lattice units, needs < 1
    steps = int(round(t[-1] / dt_l))

    # same gaussian blob as the finite-difference run
    xs = np.arange(N) * dx_l
    rho = np.exp(-((xs - 2.0) ** 2) / (2 * 0.4**2))

    f0, f1 = equilibrium(rho, u_l)
    for _ in range(steps):
        f0, f1 = collide(f0, f1, u_l, tau)
        f0, f1 = stream(f0, f1)

    return xs, density(f0, f1), u_l, steps


In [ ]:
N_LATTICE = 2**10  # <-- resolution knob

xs, rho_lbm, u_l, n_steps = lbm_solve(N_LATTICE)
print(f"N = {N_LATTICE}   u_lat = {u_l:.3f}   steps = {n_steps}")

LBM_GIF = "lbm_vs_analytic.gif"


def lbm_frames(N, sample_times, tau=TAU):
    dx_l = L / N
    dt_l = (tau - 0.5) * dx_l**2 / D
    u_l = u * dt_l / dx_l

    xs = np.arange(N) * dx_l
    rho = np.exp(-((xs - X0) ** 2) / (2 * SIGMA0**2))
    f0, f1 = equilibrium(rho, u_l)

    out, step = [], 0
    for target in (int(round(ts / dt_l)) for ts in sample_times):
        while step < target:
            f0, f1 = collide(f0, f1, u_l, tau)
            f0, f1 = stream(f0, f1)
            step += 1
        out.append(density(f0, f1))
    return xs, np.array(out), u_l, step


frame_times = t[list(gif_frames)]
xs_f, lbm_hist, u_l_f, total_steps = lbm_frames(N_LATTICE, frame_times)

fig_c, ax_c = plt.subplots(figsize=(10, 4))
(l_exact,) = ax_c.plot(x, history[gif_frames[0]], lw=4, alpha=0.4, label="analytic")
(l_lbm,) = ax_c.plot(xs_f, lbm_hist[0], lw=2, ls="--", color="crimson",
                     label=f"D1Q2 LBM, N = {N_LATTICE}")
ax_c.set_xlim(0, L)
ax_c.set_ylim(0, history.max() * 1.05)
ax_c.set_xlabel("x")
ax_c.set_ylabel("rho")
ax_c.legend(loc="upper right")
title_c = ax_c.set_title(f"t = {frame_times[0]:.2f}")
fig_c.tight_layout()


def frame_c(k):
    l_exact.set_ydata(history[gif_frames[k]])
    l_lbm.set_ydata(lbm_hist[k])
    title_c.set_text(f"t = {frame_times[k]:.2f}")
    return l_exact, l_lbm, title_c


anim_c = FuncAnimation(fig_c, frame_c, frames=len(frame_times),
                       interval=1000 / GIF_FPS, blit=True)
anim_c.save(LBM_GIF, writer=PillowWriter(fps=GIF_FPS))
plt.close(fig_c)

print(f"{LBM_GIF}: {len(frame_times)} frames")
HTML(f'<img src="{LBM_GIF}">')


In [ ]:
def lbm_lattice(rho0, u_lat, steps, tau=1.0):
    # straight lattice units: dx = dt = 1, no continuum mapping
    # u_lat is the velocity in lattice units and must satisfy |u_lat| < 1
    # populations start at equilibrium, so the first collision is a no-op at tau = 1
    f0, f1 = equilibrium(rho0, u_lat)
    hist = [density(f0, f1)]
    for _ in range(steps):
        f0, f1 = collide(f0, f1, u_lat, tau)
        f0, f1 = stream(f0, f1)
        hist.append(density(f0, f1))
    return np.array(hist)


<a id="sec-qlbm"></a>

---

## 3. Quantum LBM Circuit


In [ ]:
N_QUBITS = 2
U_LAT = 0.2
STEPS = 1

N_SITES = 2**N_QUBITS
rho_q = np.zeros(N_SITES)
rho_q[1] = 2.0
rho_q[2] = 1.0

hist_q = lbm_lattice(rho_q, U_LAT, STEPS)

sites = np.arange(N_SITES)
bar_w = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
for k, rho_k in enumerate(hist_q):
    offset = (k - 0.5) * bar_w
    ax.bar(sites + offset, rho_k, width=bar_w, label=f"step {k}")

ax.set_xticks(sites)
ax.set_xlabel("lattice site")
ax.set_ylabel("rho")
ax.legend()
plt.tight_layout()
plt.show()

for k, rho_k in enumerate(hist_q):
    print(f"step {k}: {np.round(rho_k, 4)}   mass = {rho_k.sum():.4f}")


In [ ]:
def encode(rho, u_lat):
    # amplitudes: first half is f0 (dir=0), second half is f1 (dir=1)
    # amplitude = sqrt(f / M), so probability = f / M
    M = rho.sum()
    f0, f1 = equilibrium(rho, u_lat)
    return np.concatenate([np.sqrt(f0 / M), np.sqrt(f1 / M)]), M


def increment(qc, q):
    # +1 mod 2**n on the position register
    for k in range(len(q) - 1, 0, -1):
        qc.mcx(list(q[:k]), q[k])
    qc.x(q[0])


def build_circuit(psi0, with_measurements=False):
    # psi0 holds 2 * 2**n amplitudes: n position qubits plus one direction qubit
    n = int(np.log2(len(psi0))) - 1
    qpos = QuantumRegister(n, "pos")
    qdir = QuantumRegister(1, "dir")
    if with_measurements:
        creg = ClassicalRegister(n + 1, "c")
        qc = QuantumCircuit(qpos, qdir, creg)
    else:
        qc = QuantumCircuit(qpos, qdir)

    qc.prepare_state(psi0)
    qc.barrier(label="psi0")

    # streaming. NOT(NOT(x) + 1) = x - 1, so complementing the position register
    # on the direction qubit turns a single increment into +1 for the right-movers
    # (dir = 0) and -1 for the left-movers (dir = 1)
    for qb in qpos:
        qc.cx(qdir[0], qb)
    increment(qc, qpos)
    for qb in qpos:
        qc.cx(qdir[0], qb)
    qc.barrier(label="streamed")

    # classical bit j holds position qubit j, the last bit holds the direction qubit,
    # so int(bitstring, 2) indexes the amplitudes the same way as the statevector
    if with_measurements:
        qc.measure(list(qpos) + [qdir[0]], range(n + 1))

    return qc


In [ ]:
psi0, M = encode(rho_q, U_LAT)
qc = build_circuit(psi0)

# exact readout, no measurement: f_v(x) = M |amplitude|^2
probs = np.abs(Statevector(qc).data) ** 2
rho_quantum = M * probs[:N_SITES] + M * probs[N_SITES:]

rho_ref = lbm_lattice(rho_q, U_LAT, 1)[-1]
print("classical LBM :", np.round(rho_ref, 6))
print("quantum       :", np.round(rho_quantum, 6))
print("match         :", np.allclose(rho_quantum, rho_ref))

# draw the measured version: Statevector above needs a circuit without measurements
build_circuit(psi0, with_measurements=True).draw(output="mpl")


In [ ]:
ONE_QUBIT_GATES = ["u", "u1", "u2", "u3", "rx", "ry", "rz", "sx", "x", "y", "z", "h"]
TWO_QUBIT_GATES = ["cx", "cz", "ecr"]


def make_noise_model(noise_2q):
    nm = NoiseModel()
    if noise_2q > 0:
        nm.add_all_qubit_quantum_error(
            depolarizing_error(noise_2q / 5, 1), ONE_QUBIT_GATES
        )
        nm.add_all_qubit_quantum_error(depolarizing_error(noise_2q, 2), TWO_QUBIT_GATES)
    return nm


def probs_from_counts(counts, shots, n_total):
    p = np.zeros(2**n_total)
    for bitstring, count in counts.items():
        p[int(bitstring.replace(" ", ""), 2)] = count / shots
    return p


def qlbm_trajectory(rho0, u_lat, steps, mode="statevector", shots=10_000,
                    noise_2q=0.0, seed=None):
    # hybrid loop: the circuit does the streaming, we re-encode after every step.
    # collision is dissipative, so it cannot be folded into a unitary and the
    # trajectory has to be rebuilt from the density each step.
    n_sites = len(rho0)
    n_total = int(np.log2(n_sites)) + 1
    rng = np.random.default_rng(seed)

    sim = None
    if mode != "statevector":
        nm = make_noise_model(noise_2q) if mode == "noisy" else None
        sim = AerSimulator(noise_model=nm)

    rho = rho0.astype(float).copy()
    history = [rho.copy()]
    for _ in range(steps):
        psi, M = encode(rho, u_lat)
        qc = build_circuit(psi, with_measurements=(mode != "statevector"))

        if mode == "statevector":
            probs = np.abs(Statevector(qc).data) ** 2
        else:
            job = sim.run(transpile(qc, sim), shots=shots,
                          seed_simulator=int(rng.integers(2**31)))
            probs = probs_from_counts(job.result().get_counts(), shots, n_total)

        rho = M * probs[:n_sites] + M * probs[n_sites:]
        history.append(rho.copy())

    return np.array(history)


<a id="sec-tracking"></a>

---

## 4. Experiment Tracking with MLflow


In [ ]:
# on Qubernetes MLFLOW_TRACKING_URI points at the platform server,
# locally it falls back to ./mlruns so the notebook still runs standalone
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "mlruns"))
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "demo-experiment")
mlflow.set_experiment(EXPERIMENT_NAME)

# provenance: Qubernetes injects these so every run traces back to a commit
GIT_TAGS = {
    "mlflow.source.git.commit": os.getenv("MLFLOW_GIT_COMMIT", "unknown"),
    "mlflow.source.git.branch": os.getenv("MLFLOW_GIT_BRANCH", "unknown"),
    "mlflow.source.git.repoURL": os.getenv("MLFLOW_GIT_REPO_URL", "unknown"),
}

print("tracking to:", mlflow.get_tracking_uri())


In [ ]:
def circular_fit(rho):
    # wrapped-normal parameters from the circular first moment (closed-form)
    n = len(rho)
    sites = np.arange(n)
    mass = rho.sum()
    z = (rho * np.exp(2j * np.pi * sites / n)).sum() / mass
    centre = (np.angle(z) * n / (2 * np.pi)) % n
    sigma = n / (2 * np.pi) * np.sqrt(max(-2 * np.log(abs(z)), 1e-12))
    return mass, centre, sigma


def wrapped_gaussian(at, mass, centre, sigma, period, n_images=4):
    def kernel(points):
        return sum(np.exp(-((points - centre + k * period) ** 2) / (2 * sigma**2))
                   for k in range(-n_images, n_images + 1))
    return mass * kernel(at) / kernel(np.arange(period)).sum()


def fit_gaussian(rho):
    from scipy.optimize import minimize
    n = len(rho)
    sites = np.arange(n)
    mass0, centre0, sigma0 = circular_fit(rho)

    def cost(params):
        m, c, s = params
        return np.sum((wrapped_gaussian(sites, m, c, max(s, 0.1), n) - rho) ** 2)

    res = minimize(cost, [mass0, centre0, sigma0], method="Nelder-Mead",
                   options={"xatol": 1e-6, "fatol": 1e-10, "maxiter": 500})
    m, c, s = res.x
    return m, c % n, max(abs(s), 0.1)


In [ ]:
def circuit_stats(psi0):
    full = transpile(build_circuit(psi0), basis_gates=["cx", "u"], optimization_level=1)
    prep = QuantumCircuit(int(np.log2(len(psi0))))
    prep.prepare_state(psi0)
    prep = transpile(prep, basis_gates=["cx", "u"], optimization_level=1)
    prep_cx = prep.count_ops().get("cx", 0)
    total_cx = full.count_ops().get("cx", 0)
    return {
        "circuit_depth": full.depth(),
        "cx_total": total_cx,
        "cx_state_prep": prep_cx,
        "cx_streaming": total_cx - prep_cx,
    }


In [ ]:
def log_trajectory(rho0, u_lat, steps, mode, shots=10_000, noise_2q=0.0,
                   seed=None, tau=1.0, run_name=None):
    """Run one trajectory and record it as a single MLflow run."""
    n_sites = len(rho0)
    n_position = int(np.log2(n_sites))
    reference = lbm_lattice(rho0, u_lat, steps, tau=tau)

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags({**GIT_TAGS, "mode": mode})
        mlflow.log_params({
            "mode": mode,
            "n_position": n_position,
            "n_qubits": n_position + 1,
            "n_sites": n_sites,
            "u_lat": u_lat,
            "tau": tau,
            "steps": steps,
            "shots": 0 if mode == "statevector" else shots,
            "noise_2q": noise_2q,
            "noise_1q": noise_2q / 5,
            "seed": seed,
        })

        psi0, _ = encode(rho0, u_lat)
        mlflow.log_metrics(circuit_stats(psi0))

        history = qlbm_trajectory(rho0, u_lat, steps, mode=mode, shots=shots,
                                  noise_2q=noise_2q, seed=seed)

        errors = np.zeros(len(history))
        errors_fit = np.zeros(len(history))
        lattice = np.arange(n_sites)
        for k, (rho_k, ref_k) in enumerate(zip(history, reference)):
            errors[k] = np.linalg.norm(rho_k - ref_k)

            mass_k, fit_centre, fit_width = fit_gaussian(rho_k)
            fitted = wrapped_gaussian(lattice, mass_k, fit_centre, fit_width, n_sites)
            errors_fit[k] = np.linalg.norm(fitted - ref_k)  

            mlflow.log_metric("l2_error", round(float(errors[k]), 5), step=k)

        mlflow.log_metrics({
            "final_l2_error": round(float(errors[-1]), 5),
            "mean_l2_error": round(float(errors.mean()), 5),
            "final_l2_error_fitted": round(float(errors_fit[-1]), 5),
            "mean_l2_error_fitted": round(float(errors_fit.mean()), 5),
        })
        mlflow.log_dict({"history": history.tolist(),
                         "reference": reference.tolist()}, "density_history.json")

    return history, errors


In [ ]:
# the trajectory demo configuration
N_POS_TRAJ = 5         # position qubits -> 32 sites
STEPS_TRAJ = 40
U_TRAJ = 0.4
N_SITES_TRAJ = 2**N_POS_TRAJ

_sites = np.arange(N_SITES_TRAJ)
RHO0 = 2.0 * np.exp(-((_sites - N_SITES_TRAJ // 4) ** 2) / (2 * 1.5**2))

# smooth start matters: D1Q2 at tau=1 decouples odd and even sites, so a sharp
# spike would show a checkerboard comb


<a id="sec-sweep"></a>

---

## 5. Sweep & Analysis


In [ ]:
SWEEP_SHOTS = [1_000, 10_000, 100_000]
SWEEP_NOISE = [0.0, 1e-5, 1e-4, 1e-3]  # 0 is the shot-noise-only baseline
SWEEP_REPEATS = 3

# full grid, so every noisy run has a noiseless baseline at the same shot count
runs = {}
runs["statevector"] = log_trajectory(
    RHO0, U_TRAJ, STEPS_TRAJ, "statevector", run_name="statevector"
)

for shots in SWEEP_SHOTS:
    for noise in SWEEP_NOISE:
        for r in range(SWEEP_REPEATS):
            runs[(shots, noise, r)] = log_trajectory(
                RHO0, U_TRAJ, STEPS_TRAJ,
                mode="shots" if noise == 0 else "noisy",
                shots=shots, noise_2q=noise, seed=100 + r,
                run_name=f"s{shots}-n{noise:g}-r{r}",
            )

print(f"logged {len(runs)} runs to {mlflow.get_tracking_uri()}")


In [ ]:
TRAJ_GIF = "qlbm_trajectory.gif"
HI_SHOTS, LO_SHOTS = 100_000, 1_000

sv_hist = runs["statevector"][0]
hi_hist = runs[(HI_SHOTS, 0.0, 0)][0]
lo_hist = runs[(LO_SHOTS, 0.0, 0)][0]

sites = np.arange(N_SITES_TRAJ)
ymax = max(sv_hist.max(), hi_hist.max(), lo_hist.max()) * 1.1

traj_fig, traj_ax = plt.subplots(figsize=(10, 4.5))
(l_exact,) = traj_ax.plot(sites, sv_hist[0], lw=3, color="tab:blue",
                          label="statevector (exact)")
(l_hi,) = traj_ax.plot(sites, hi_hist[0], "o", ms=6, color="tab:green",
                       label=f"{HI_SHOTS:,} shots")
(l_lo,) = traj_ax.plot(sites, lo_hist[0], "^", ms=6, color="tab:red",
                       label=f"{LO_SHOTS:,} shots")
traj_ax.set_xlim(0, N_SITES_TRAJ - 1)
traj_ax.set_ylim(0, ymax)
traj_ax.set_xlabel("lattice site")
traj_ax.set_ylabel("rho")
traj_ax.legend(loc="upper right")
traj_title = traj_ax.set_title("step 0")
traj_fig.tight_layout()


def traj_frame(k):
    l_exact.set_ydata(sv_hist[k])
    l_hi.set_ydata(hi_hist[k])
    l_lo.set_ydata(lo_hist[k])
    traj_title.set_text(f"step {k}")
    return l_exact, l_hi, l_lo, traj_title


traj_anim = FuncAnimation(traj_fig, traj_frame, frames=len(sv_hist),
                         interval=150, blit=True)
traj_anim.save(TRAJ_GIF, writer=PillowWriter(fps=6))
plt.close(traj_fig)

print(f"wrote {TRAJ_GIF} ({len(sv_hist)} frames)")
HTML(f'<img src="{TRAJ_GIF}">')


In [ ]:
def error_curves(shots, noise):
    """L2 error against the classical trajectory, shaped (repeats, steps).

    runs[key] is (density_history, errors); errors[k] is the L2 distance at step k.
    Step 0 is dropped because both start from the same profile, so it is exactly 0.
    """
    return np.array([runs[(shots, noise, r)][1][1:] for r in range(SWEEP_REPEATS)])


SHOT_COLOURS = dict(zip(SWEEP_SHOTS,
                        plt.cm.viridis(np.linspace(0.15, 0.85, len(SWEEP_SHOTS)))))

FIG_STEPS = "error_vs_step.png"
step_axis = np.arange(1, STEPS_TRAJ + 1)

fig_steps, ax = plt.subplots(figsize=(10, 5.5))
for shots in SWEEP_SHOTS:
    colour = SHOT_COLOURS[shots]
    # noise_2q = 0, so finite sampling is the only error source here
    curves = error_curves(shots, 0.0)
    ax.plot(step_axis, curves.mean(axis=0), lw=2.5, color=colour,
            label=f"{shots:,} shots")
    ax.fill_between(step_axis, curves.min(axis=0), curves.max(axis=0),
                    color=colour, alpha=0.15)

ax.set_yscale("log")
ax.set_xlabel("LBM step")
ax.set_ylabel("L2 error vs classical")
ax.set_title("Cumulative error across timesteps")
ax.legend()

plt.tight_layout()
fig_steps.savefig(FIG_STEPS, dpi=130, bbox_inches="tight")
plt.show()

# if the per-step errors simply accumulated they would grow like sqrt(steps)
print(f"independent accumulation over {STEPS_TRAJ} steps would grow "
      f"{np.sqrt(STEPS_TRAJ):.1f}x. observed:")
for shots in SWEEP_SHOTS:
    c = error_curves(shots, 0.0).mean(axis=0)
    print(f"  {shots:>7,} shots: {c[-10:].mean() / c[:10].mean():.2f}x")


In [ ]:
def summary(shots, noise):
    # average each repeat over the trajectory, then aggregate across repeats
    per_repeat = error_curves(shots, noise).mean(axis=1)
    return per_repeat.mean(), per_repeat.std()


FIG_SWEEPS = "error_vs_noise.png"
nonzero = [v for v in SWEEP_NOISE if v > 0]

fig_sweeps, ax_n = plt.subplots(figsize=(9.5, 6))
for shots in SWEEP_SHOTS:
    colour = SHOT_COLOURS[shots]
    m, s = np.array([summary(shots, v) for v in nonzero]).T
    ax_n.errorbar(nonzero, m, yerr=s, fmt="o-", ms=9, lw=2.5, capsize=5,
                  color=colour, label=f"{shots:,} shots")
    # its own shot-noise floor, same colour: zero cannot sit on a log axis
    floor, _ = summary(shots, 0.0)
    ax_n.axhline(floor, ls=":", lw=2, color=colour, alpha=0.9)

ax_n.set_xscale("log")
ax_n.set_yscale("log")
ax_n.set_xlabel("two-qubit gate error rate")
ax_n.set_ylabel("mean L2 error over trajectory")
ax_n.set_title("Scaling of error with gate noise")

# name the dotted lines in the legend, not in the title
handles, labels = ax_n.get_legend_handles_labels()
handles.append(plt.Line2D([], [], ls=":", lw=2, color="gray"))
labels.append("shot noise floor")
ax_n.legend(handles, labels)

plt.tight_layout()
fig_sweeps.savefig(FIG_SWEEPS, dpi=130, bbox_inches="tight")
plt.show()

print(f"{'shots':>8} {'floor':>9} " + " ".join(f"{v:>9.0e}" for v in nonzero))
for shots in SWEEP_SHOTS:
    row = " ".join(f"{summary(shots, v)[0]:9.4f}" for v in nonzero)
    print(f"{shots:>8,} {summary(shots, 0.0)[0]:9.4f} {row}")


In [ ]:
FIT_GIF = "qlbm_gaussian_fit.gif"
FIT_SHOTS = 1_000

noisy_hist = runs[(FIT_SHOTS, 0.0, 0)][0]
ref_hist = lbm_lattice(RHO0, U_TRAJ, STEPS_TRAJ)

lattice = np.arange(N_SITES_TRAJ)
smooth = np.linspace(0, N_SITES_TRAJ, 400, endpoint=False)
fits = [fit_gaussian(r) for r in noisy_hist]

fig_g, ax_g = plt.subplots(figsize=(10, 4.6))
(l_ref,) = ax_g.plot(lattice, ref_hist[0], lw=5, alpha=0.3, color="tab:blue",
                     label="classical LBM")
(m_raw,) = ax_g.plot(lattice, noisy_hist[0], "o", ms=5, color="tab:red",
                     label=f"{FIT_SHOTS:,} shots")
(l_fit,) = ax_g.plot(smooth, wrapped_gaussian(smooth, *fits[0], N_SITES_TRAJ),
                     lw=2.5, color="tab:purple", label="gaussian fit")
ax_g.set_xlim(0, N_SITES_TRAJ - 1)
ax_g.set_ylim(0, noisy_hist.max() * 1.1)
ax_g.set_xlabel("lattice site")
ax_g.set_ylabel("rho")
ax_g.legend(loc="upper right")
title_g = ax_g.set_title("step 0")
fig_g.tight_layout()


def frame_g(k):
    l_ref.set_ydata(ref_hist[k])
    m_raw.set_ydata(noisy_hist[k])
    l_fit.set_ydata(wrapped_gaussian(smooth, *fits[k], N_SITES_TRAJ))
    title_g.set_text(f"step {k}")
    return l_ref, m_raw, l_fit, title_g


anim_g = FuncAnimation(fig_g, frame_g, frames=len(noisy_hist),
                       interval=1000 / 6, blit=True)
anim_g.save(FIT_GIF, writer=PillowWriter(fps=6))
plt.close(fig_g)

print(f"{FIT_GIF}: {len(noisy_hist)} frames")
HTML(f'<img src="{FIT_GIF}">')


In [ ]:
def raw_and_fitted(shots):
    hist = runs[(shots, 0.0, 0)][0]
    raw = np.array([np.linalg.norm(hist[k] - ref_hist[k])
                    for k in range(1, len(hist))])
    fit = np.array([np.linalg.norm(
        wrapped_gaussian(lattice, *fit_gaussian(hist[k]), N_SITES_TRAJ) - ref_hist[k])
        for k in range(1, len(hist))])
    return raw.mean(), fit.mean()


FIT_TABLE = {s: raw_and_fitted(s) for s in SWEEP_SHOTS}

print(f"{'shots':>9} {'raw':>9} {'fitted':>9} {'gain':>7}")
for shots, (raw, fit) in FIT_TABLE.items():
    print(f"{shots:>9,} {raw:9.4f} {fit:9.4f} {raw / fit:6.2f}x")


In [ ]:
CSV_SUMMARY = "sweep_summary.csv"

table = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])
table = table[table.get("tags.mode", "") != "summary"]

with mlflow.start_run(run_name="sweep-summary"):
    mlflow.set_tags({**GIT_TAGS, "mode": "summary"})
    mlflow.log_params({
        "n_position": N_POS_TRAJ,
        "n_qubits": N_POS_TRAJ + 1,
        "n_sites": N_SITES_TRAJ,
        "steps": STEPS_TRAJ,
        "u_lat": U_TRAJ,
        "sweep_shots": SWEEP_SHOTS,
        "sweep_noise": SWEEP_NOISE,
        "repeats": SWEEP_REPEATS,
        "runs_aggregated": len(table),
    })

    table.to_csv(CSV_SUMMARY, index=False)
    for path in (CSV_SUMMARY, FIG_STEPS, FIG_SWEEPS, TRAJ_GIF, FIT_GIF):
        mlflow.log_artifact(path)

print(f"summary run aggregates {len(table)} runs")
